In [0]:
dbutils.widgets.text("query", "Spirit Airlines")
dbutils.widgets.text("city", "New York")
dbutils.widgets.text("radius", "100")
dbutils.widgets.text("catalog", "rohitb_demo")
dbutils.widgets.text("schema", "spirit_demo")
dbutils.widgets.text("table_name", "google_reviews")

In [0]:
def get_lat_long(location):
    from pyspark.sql.functions import variant_get
    response_format = """
    {
      "type": "json_schema",
      "json_schema": {
        "name": "Location",
        "schema": {
          "type": "object",
          "properties": {
            "latitude": {"type": "number"},
            "longitude": {"type": "number"}
          }
        }
      }
    }
    """

    df = spark.sql(
      f"""
      select
      parse_json
        (ai_query(
          'databricks-meta-llama-3-3-70b-instruct',
          'What is the latitude and longitude of {location}, estimates are fine.',
          responseFormat =>'{response_format}'
        )
        ) as location
        """
    ).withColumn('latitude', variant_get('location','$.latitude','float')).withColumn('longitude', variant_get('location','$.longitude','float'))

    lat, long = df.collect()[0]['latitude'], df.collect()[0]['longitude']

    return lat, long

In [0]:
query = dbutils.widgets.get("query")
# location = '39.000000,-75.500000.'
city = dbutils.widgets.get("city")
location = get_lat_long(city)
radius = dbutils.widgets.get("radius")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
table_name = dbutils.widgets.get("table_name")

In [0]:
print(query, location, radius, catalog, schema, table_name)

In [0]:
def get_google_reviews(query, location, radius):
    import requests
    import csv
    import time

    API_KEY = dbutils.secrets.get(scope='rohitb_demo', key='google_api_key')

    def get_place_ids(query, location, radius):
        url = f"https://maps.googleapis.com/maps/api/place/textsearch/json?query={query}&location={location}&radius={radius}&key={API_KEY}"
        response = requests.get(url)
        places = response.json().get('results', [])
        place_details = [(place['place_id'], place['name'], place['formatted_address']) for place in places]
        return place_details

    def get_reviews(place_id):
        url = f'https://maps.googleapis.com/maps/api/place/details/json?place_id={place_id}&key={API_KEY}'
        response = requests.get(url)
        reviews = response.json().get('result', {}).get('reviews', [])
        return reviews

    def fetch_reviews_for_all_places(query, location, radius):
        place_details = get_place_ids(query, location, radius)
        all_reviews = []

        for place_id, name, address in place_details:
            reviews = get_reviews(place_id)
            for review in reviews:
                all_reviews.append([place_id, name, address, review['author_name'], review['rating'], review['text'], review['relative_time_description']])
            time.sleep(2)  # Pause to respect API rate limits

        return all_reviews
    
    return fetch_reviews_for_all_places(query, location, radius)

In [0]:
get_lat_long(city)

In [0]:
all_reviews = get_google_reviews(query, location, radius)

In [0]:
from pyspark.sql.functions import current_date, monotonically_increasing_id

df = spark.createDataFrame(all_reviews, ['place_id', 'name', 'address', 'author', 'rating', 'review', 'time'])
df = df.withColumn('load_date', current_date()).withColumn('review_id', monotonically_increasing_id())

In [0]:
df.write.mode("append").saveAsTable(f"{catalog}.{schema}.{table_name}", ifNotExists=True)